In [29]:
import accelforge as af
from scheduling.scheduler import *
from af_wrapper import *
import numpy as np


def run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = None,
    total_latency_grid = None,
    actions_grid = None,
    memory_name = None,
    shared_memory_info = None,
):
    schedule, min_latency = best_schedule(
        einsums,
        compute_units,
        shared_memory_info,
        data_dependencies,
        latency_per_component_grid,
        total_latency_grid,
        actions_grid,
        memory_name
    )
    return schedule, min_latency


In [30]:
arch = "arch/eyeriss-dual-core-identical/full.yaml"
m2_workload = "workload/mem-partition-tests/parallel-mm/full.yaml"

In [13]:
spec = af.Spec.from_yaml(
    arch,
    m2_workload,
    jinja_parse_data=dict(M=256, N=256, K=128, X=256, Z=256, Y=128)
)

In [14]:
m2_full_mapping = af_map(
    "arch/eyeriss-dual-core-identical/full.yaml",
    "workload/mem-partition-tests/parallel-mm/full.yaml",
    jinja_parse_data=dict(M=256, N=256, K=128, X=256, Z=256, Y=128)
)

Getting energy, latency, and leak power for components running each Einsum. : 100%|██████████| 2/2 [00:05<00:00,  2.90s/it]
Generating pmapping templates for compute MAC2 Einsum E0: 16it [00:00, 21.15it/s]
Generating pmapping templates for compute MAC2 Einsum E1: 16it [00:00, 20.08it/s]
Generating pmapping templates for compute MAC1 Einsum E0: 16it [00:00, 21.56it/s]
Generating pmapping templates for compute MAC1 Einsum E1: 16it [00:00, 20.05it/s]
Generating jobs: 100%|██████████| 2/2 [00:02<00:00,  1.13s/it]


Einsum E0 has 32 pmapping jobs:
	0	[C in MainMemory] [B in MainMemory] [A in MainMemory] T-m  T-n  S-reuse_output-n  S-reuse_weight-m  [A in InputScratchpad2] [B in WeightScratchpad2] T-m  [C in OutputScratchpad2] T-n  MAC2 computes E0
	1	[C in MainMemory] [B in MainMemory] [A in MainMemory] T-m  T-n  [A in GlobalBuffer] S-reuse_output-n  S-reuse_weight-m  [A in InputScratchpad2] [B in WeightScratchpad2] T-m  [C in OutputScratchpad2] T-n  MAC2 computes E0
	2	[C in MainMemory] [B in MainMemory] [A in MainMemory] T-n  [B in GlobalBuffer] T-m  S-reuse_output-n  S-reuse_weight-m  [A in InputScratchpad2] [B in WeightScratchpad2] T-m  [C in OutputScratchpad2] T-n  MAC2 computes E0
	3	[C in MainMemory] [B in MainMemory] [A in MainMemory] T-m  [C in GlobalBuffer] T-n  S-reuse_output-n  S-reuse_weight-m  [A in InputScratchpad2] [B in WeightScratchpad2] T-m  [C in OutputScratchpad2] T-n  MAC2 computes E0
	4	[C in MainMemory] [B in MainMemory] [A in MainMemory] T-m  T-n  [A in GlobalBuffer] [B in

Compressing pmappings: 100%|██████████| 2/2 [00:00<00:00, 54.98it/s]


Dirty joining with resource usage <= 1.2× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████| 2/2 [00:00<00:00, 80.00it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>latency=1.24e-04    Total<SEP>energy=7.26e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████| 2/2 [00:00<00:00, 81.33it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 2 -> 2 (100.00% kept) pmappings


Final consolidate: 100%|██████████| 1/1 [00:00<00:00, 573.46it/s]


Dirty joining mapping(s) valid & optimal! Returning...


In [6]:
mappings = []
for i in range(1):
    filename = trunc(arch) + "-" + trunc(m2_workload) + "-" + str(i)
    with open("images/" + filename + ".svg", "w") as f:
        f.write(m2_full_mapping[i].render())
    mappings.append(process_mapping(arch, m2_workload, m2_full_mapping[i]))

mappings

[{'mapping': <accelforge.mapper.FFM.mappings.Mappings at 0x40765b2300>,
  'total reads': np.float64(1310720.0),
  'total write': np.float64(786432.0),
  'total r+w': np.float64(2097152.0),
  'latency': np.float32(0.001772247),
  '(r+w)/latency': np.float64(1183329397.0567572),
  'bw util': np.float64(0.18489521829011832),
  'mem summary': {('E0', 'MainMemory'): np.float64(1048576.0),
   ('E1', 'MainMemory'): np.float64(1048576.0)}}]

In [15]:
data_dependencies = {
    "E0": [],
    "E1": [],
}
compute_units = ['core2', 'core1']
einsums = data_dependencies.keys()
memory_name = "MainMemory"
shared_mem_info = {'GlobalBuffer' : [(0, 100), (100, 0), (50, 50), (25, 75), (75, 25)]}

In [16]:
from arch.arch_utils import *

arch_pairings = generate_architecture_pairings(compute_units, shared_mem_info)

In [42]:
%%capture grid_output

# RUN TO GENERATE ACCELFORGE VALUES.
# To skip recomputation when running with fresh kernel, comment out
# this cell and use the below assignments instead.

(grid_lats, grid_mems, grid_maps) = af_memoizable_grid_mem(
    einsums, 
    arch_pairings,
    lambda einsum: "workload/mem-partition-tests/parallel-mm/"+einsum+".yaml",
    "arch/eyeriss-dual-core-identical/",
    jinja_parse_data=dict(M=128, N=1024, K=128, X=128, Z=128, Y=1024)
)

In [43]:
grid_lats

{(('core2', ('GlobalBuffer', 0)), 'E0'): {'MAC2': np.float32(0.00131072),
  'InputScratchpad2': np.float32(0.00074227224),
  'MainMemory': np.float32(0.0040234374),
  'WeightScratchpad2': np.float32(0.00026763324),
  'OutputScratchpad2': np.float32(0.0018297874)},
 (('core2', ('GlobalBuffer', 0)), 'E1'): {'MAC2': np.float32(0.00131072),
  'InputScratchpad2': np.float32(0.00074227224),
  'MainMemory': np.float32(0.0040234374),
  'WeightScratchpad2': np.float32(0.00026763324),
  'OutputScratchpad2': np.float32(0.0018297874)},
 (('core2', ('GlobalBuffer', 25)), 'E0'): {'MAC2': np.float32(0.00131072),
  'InputScratchpad2': np.float32(0.00074227224),
  'MainMemory': np.float32(0.0006640625),
  'WeightScratchpad2': np.float32(0.00042821318),
  'GlobalBuffer': np.float32(4.864e-05),
  'OutputScratchpad2': np.float32(0.0015535931)},
 (('core2', ('GlobalBuffer', 25)), 'E1'): {'MAC2': np.float32(0.00131072),
  'InputScratchpad2': np.float32(0.00074227224),
  'MainMemory': np.float32(0.0006640625

In [38]:
grid_mems

{(('core2', ('GlobalBuffer', 0)),
  'E0'): {('InputScratchpad2',
   'read'): np.float64(67108864.0), ('InputScratchpad2', 'write'): np.float32(524288.0), ('MainMemory',
   'read'): np.float32(4.58752e+06), ('MainMemory',
   'write'): np.float64(2097152.0), ('WeightScratchpad2',
   'read'): np.float64(67108864.0), ('WeightScratchpad2',
   'write'): np.float32(1.6777216e+07), ('OutputScratchpad2',
   'read'): np.float32(8.28375e+07), ('OutputScratchpad2',
   'write'): np.float32(8.28375e+07), ('MAC2',
   'compute'): np.float64(8388608.0)},
 (('core2', ('GlobalBuffer', 0)),
  'E1'): {('InputScratchpad2',
   'read'): np.float64(67108864.0), ('InputScratchpad2',
   'write'): np.float32(524288.0), ('MainMemory',
   'read'): np.float32(4.58752e+06), ('MainMemory',
   'write'): np.float64(2097152.0), ('WeightScratchpad2',
   'read'): np.float64(67108864.0), ('WeightScratchpad2',
   'write'): np.float32(1.6777216e+07), ('OutputScratchpad2',
   'read'): np.float32(8.28375e+07), ('OutputScratchpa

In [39]:
# for (key, mapping) in grid_maps.items():
#     print(mapping.latency(per_component=True))
#     print("Overall latency:", mapping.latency(per_einsum=True))
#     print()
#     print()

In [40]:
schedule, latency = run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = grid_lats,
    actions_grid = grid_mems,
    memory_name = memory_name,
    shared_memory_info = shared_mem_info
)

In [41]:
schedule

{(E0, ('core2', ('GlobalBuffer', 25)), latency=0.0007710424833931029): 0,
 (E1, ('core2', ('GlobalBuffer', 25)), latency=0.0007710424833931029): 0}

In [14]:
latency

np.float64(0.00038905909059394617)

In [15]:
mems = sum(
    sum(
        count
        for (action, count) in grid_mems[(node.compute_unit, node.einsum_name)].items()
        if action[0] == memory_name
    ) 
    for node in schedule.keys()
)
mems

np.float64(1185792.0)

In [16]:
# bwu
(mems/latency) / 6.4e9 # old schedule: np.float64(0.4762258443913986)

np.float64(0.47622585997707306)